In [1]:
import jax
import jax.numpy as jnp
import jax.random as random
from jax import lax, jit, vmap
from jax.experimental.pjit import pjit
from jax.experimental import mesh_utils
from jax.sharding import Mesh, PartitionSpec, PositionalSharding
from functools import partial

# --- Configuration
MAX_RECURSION_DEPTH = 50
DIMENSIONAL_CONSTRAINT = 0.8
RECURSION_DEPTHS = [5, 8, 10, 15]

@jit
def dynamic_pi(depth, scale_factor):
    depth = jnp.minimum(depth, MAX_RECURSION_DEPTH)
    return jnp.pi * jnp.log1p(depth + 1) * scale_factor * DIMENSIONAL_CONSTRAINT

@jit
def dynamic_phi(depth, scale_factor):
    depth = jnp.minimum(depth, MAX_RECURSION_DEPTH)
    return (1 + jnp.sqrt(5)) / 2 * jnp.exp(-depth / (scale_factor + 1)) * DIMENSIONAL_CONSTRAINT

@partial(jit, static_argnames=["depth"])
def dppu_with_dynamic_pi_phi(x, depth=10, scale_factor=1.0):
    depth = jnp.minimum(depth, MAX_RECURSION_DEPTH)

    def body_fn(i, val):
        pi_dyn = dynamic_pi(i, scale_factor)
        phi_dyn = dynamic_phi(i, scale_factor)
        scale = jnp.log1p(i + 1) * scale_factor * DIMENSIONAL_CONSTRAINT

        new_val = jnp.sin(val * scale * pi_dyn) * jnp.exp(-val / (phi_dyn + 1))
        # NO threshold, NO mask, NO sign, NO boolean logic
        return new_val

    return lax.fori_loop(0, depth, body_fn, x)

# --- Sharding Setup
devices = mesh_utils.create_device_mesh((8,))
sharding = PositionalSharding(devices)

batch_size = 50_000
data_size = 50_000
batch_input = jnp.linspace(0, 10, data_size)
batch_input = jax.device_put(batch_input, sharding)

batched_dppu_processing = pjit(
    lambda arr, d: vmap(
        lambda xi: dppu_with_dynamic_pi_phi(xi, depth=d, scale_factor=0.5), in_axes=0
    )(arr),
    in_shardings=(sharding, None),
    out_shardings=sharding,
)

# --- Run
for depth in RECURSION_DEPTHS:
    output_batch = batched_dppu_processing(batch_input, depth)
    print(f"Batch Output Shape (Depth={depth}):", output_batch.shape)

import time

NUM_TRIALS = 10
INPUT_SIZE = 50_000

# Warm-up compile
_ = dppu_with_dynamic_pi_phi(jnp.ones((INPUT_SIZE,)), depth=10)

for depth in RECURSION_DEPTHS:
    times = []
    for _ in range(NUM_TRIALS):
        start = time.time()
        result = dppu_with_dynamic_pi_phi(jnp.ones((INPUT_SIZE,)), depth=depth)
        _ = jax.device_get(result)
        end = time.time()
        times.append(end - start)

    avg_time = sum(times) / len(times)
    print(f"\n🔥 TPU Benchmark (Depth={depth}, Size={INPUT_SIZE})")
    print(f"Avg: {avg_time:.6f}, Min: {min(times):.6f}, Max: {max(times):.6f}")

jax.devices()




/usr/local/lib/python3.11/dist-packages/jax/_src/core.py:701: FutureWarning: unhashable type: <class 'jax._src.interpreters.partial_eval.DynamicJaxprTracer'>. Attempting to hash a tracer will lead to an error in a future JAX release.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/jax/_src/core.py:701: FutureWarning: unhashable type: <class 'jax._src.interpreters.partial_eval.DynamicJaxprTracer'>. Attempting to hash a tracer will lead to an error in a future JAX release.
  warnings.warn(


Batch Output Shape (Depth=5): (50000,)
Batch Output Shape (Depth=8): (50000,)
Batch Output Shape (Depth=10): (50000,)
Batch Output Shape (Depth=15): (50000,)

🔥 TPU Benchmark (Depth=5, Size=50000)
Avg: 0.010008, Min: 0.001085, Max: 0.085385

🔥 TPU Benchmark (Depth=8, Size=50000)
Avg: 0.009901, Min: 0.001020, Max: 0.083555

🔥 TPU Benchmark (Depth=10, Size=50000)
Avg: 0.001674, Min: 0.001138, Max: 0.002157

🔥 TPU Benchmark (Depth=15, Size=50000)
Avg: 0.009770, Min: 0.001024, Max: 0.082015


[TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0),
 TpuDevice(id=1, process_index=0, coords=(0,0,0), core_on_chip=1),
 TpuDevice(id=2, process_index=0, coords=(1,0,0), core_on_chip=0),
 TpuDevice(id=3, process_index=0, coords=(1,0,0), core_on_chip=1),
 TpuDevice(id=4, process_index=0, coords=(0,1,0), core_on_chip=0),
 TpuDevice(id=5, process_index=0, coords=(0,1,0), core_on_chip=1),
 TpuDevice(id=6, process_index=0, coords=(1,1,0), core_on_chip=0),
 TpuDevice(id=7, process_index=0, coords=(1,1,0), core_on_chip=1)]